# Revisi — HAM10000



In [1]:


!pip install -q statsmodels imagehash

import os, json, glob, hashlib, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, EfficientNetB0, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as pp_resnet
from tensorflow.keras.applications.efficientnet import preprocess_input as pp_efficientnet
from tensorflow.keras.applications.vgg16 import preprocess_input as pp_vgg16

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import norm

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Tesis_HAM10000'
MODEL_DIR   = f'{BASE_DIR}/models'
RESULTS_DIR = f'{BASE_DIR}/results'
REVISI_DIR  = f'{BASE_DIR}/revisi_reviewer'
LEAK_DIR    = f'{REVISI_DIR}/leakage_experiment'
for d in [REVISI_DIR, LEAK_DIR]:
    os.makedirs(d, exist_ok=True)

MODELS = ['ResNet50', 'EfficientNetB0', 'VGG16']
MODEL_CONFIGS = {
    'ResNet50':       {'fn': ResNet50,       'preprocess': pp_resnet},
    'EfficientNetB0': {'fn': EfficientNetB0, 'preprocess': pp_efficientnet},
    'VGG16':          {'fn': VGG16,          'preprocess': pp_vgg16},
}
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
LABEL2IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
K = len(CLASS_NAMES)
IMG_SIZE, BATCH_SIZE, AUTOTUNE = (224, 224), 32, tf.data.AUTOTUNE
ALPHA = 0.05

Mounted at /content/drive


In [2]:

# 1. DATA & SPLIT —

import kagglehub
dataset_path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

metadata_csv = glob.glob(os.path.join(dataset_path, "**", "*metadata*.csv"), recursive=True)[0]
image_path_map = {}
for d in glob.glob(os.path.join(dataset_path, "**", "*"), recursive=True):
    if os.path.isdir(d):
        for fname in os.listdir(d):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_path_map[os.path.splitext(fname)[0]] = os.path.join(d, fname)

df = pd.read_csv(metadata_csv)
df['path'] = df['image_id'].map(image_path_map)
df = df.dropna(subset=['path']).reset_index(drop=True)
df['label'] = df['dx'].map(LABEL2IDX)

lesion_df = df.groupby('lesion_id')['label'].agg(lambda x: x.mode()[0]).reset_index()
train_lesions, temp_lesions = train_test_split(
    lesion_df, test_size=0.30, stratify=lesion_df['label'], random_state=SEED)
val_lesions, test_lesions = train_test_split(
    temp_lesions, test_size=0.50, stratify=temp_lesions['label'], random_state=SEED)

train_df = df[df['lesion_id'].isin(train_lesions['lesion_id'])].reset_index(drop=True)
val_df   = df[df['lesion_id'].isin(val_lesions['lesion_id'])].reset_index(drop=True)
test_df  = df[df['lesion_id'].isin(test_lesions['lesion_id'])].reset_index(drop=True)


def fingerprint(ids):
    return hashlib.md5(','.join(sorted(map(str, ids))).encode()).hexdigest()


split_info = {name: {'n_images': int(len(d)),
                     'n_lesions': int(d['lesion_id'].nunique()),
                     'md5_image_ids': fingerprint(d['image_id'])}
              for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]}

split_path = f'{REVISI_DIR}/split_fingerprint.json'
if os.path.exists(split_path):
    saved = json.load(open(split_path))
    assert saved == split_info, ("SPLIT BERBEDA dari run sebelumnya! "
                                 "Tabel IV dan McNemar tidak boleh dihitung dari split berbeda.")
    print("✔ Split identik dengan run sebelumnya (fingerprint cocok).")
else:
    json.dump(split_info, open(split_path, 'w'), indent=2)
    print("Fingerprint split disimpan:", split_path)
test_df[['image_id', 'lesion_id', 'dx']].to_csv(f'{REVISI_DIR}/daftar_citra_test.csv', index=False)
print(json.dumps(split_info, indent=2))

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
✔ Split identik dengan run sebelumnya (fingerprint cocok).
{
  "train": {
    "n_images": 6981,
    "n_lesions": 5229,
    "md5_image_ids": "cc70e78ab3299df51762a4b4ee00f606"
  },
  "val": {
    "n_images": 1532,
    "n_lesions": 1120,
    "md5_image_ids": "44966b17d4f55ac99b12430b5e388d2b"
  },
  "test": {
    "n_images": 1502,
    "n_lesions": 1121,
    "md5_image_ids": "c117f6e880c59f2f348276689b35a127"
  }
}


In [3]:
split_ids = pd.concat([
    train_df[['image_id', 'lesion_id', 'dx']].assign(split='train'),
    val_df[['image_id', 'lesion_id', 'dx']].assign(split='val'),
    test_df[['image_id', 'lesion_id', 'dx']].assign(split='test'),
], ignore_index=True)
split_ids.to_csv(f'{REVISI_DIR}/split_lesion_level_ids.csv', index=False)

split_md5 = {name: {'n_images': int(len(d)),
                    'n_lesions': int(d['lesion_id'].nunique()),
                    'md5_image_ids': fingerprint(d['image_id']),
                    'md5_lesion_ids': fingerprint(d['lesion_id'].unique())}
             for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]}
json.dump(split_md5, open(f'{REVISI_DIR}/split_md5_lengkap.json', 'w'), indent=2)


assert split_ids.groupby('lesion_id')['split'].nunique().max() == 1
print("Tersimpan: split_lesion_level_ids.csv dan split_md5_lengkap.json")
print(json.dumps(split_md5, indent=2))


Tersimpan: split_lesion_level_ids.csv dan split_md5_lengkap.json
{
  "train": {
    "n_images": 6981,
    "n_lesions": 5229,
    "md5_image_ids": "cc70e78ab3299df51762a4b4ee00f606",
    "md5_lesion_ids": "4096220561879ff54b546a0a28ec2387"
  },
  "val": {
    "n_images": 1532,
    "n_lesions": 1120,
    "md5_image_ids": "44966b17d4f55ac99b12430b5e388d2b",
    "md5_lesion_ids": "e7eeec7f22a23e55b9d03d3e87d7aded"
  },
  "test": {
    "n_images": 1502,
    "n_lesions": 1121,
    "md5_image_ids": "c117f6e880c59f2f348276689b35a127",
    "md5_lesion_ids": "a57f1d057d33b3bf66ce5f06ce55fa41"
  }
}


In [ ]:

# 2. PREDIKSI PER CITRA — SATU SUMBER KEBENARAN untuk Tabel IV & McNemar  [R1]

def load_and_decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label


def make_eval_ds(dataframe, preprocess_fn):
    labels = tf.keras.utils.to_categorical(dataframe['label'].values, num_classes=K)
    ds = tf.data.Dataset.from_tensor_slices((dataframe['path'].values, labels))
    ds = ds.map(load_and_decode, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


PRED_CSV = f'{REVISI_DIR}/prediksi_per_citra_test.csv'
if os.path.exists(PRED_CSV):
    pred_df = pd.read_csv(PRED_CSV)
    print("Prediksi per citra dimuat dari:", PRED_CSV)
else:
    pred_df = test_df[['image_id', 'lesion_id', 'dx', 'label']].rename(columns={'label': 'y_true'})
    for m in MODELS:
        print(f"Memprediksi {m} ...")
        model = tf.keras.models.load_model(f'{MODEL_DIR}/final_{m}.keras')
        probs = model.predict(make_eval_ds(test_df, MODEL_CONFIGS[m]['preprocess']), verbose=0)
        assert len(probs) == len(test_df)
        pred_df[f'pred_{m}'] = probs.argmax(axis=1)
        for i, c in enumerate(CLASS_NAMES):
            pred_df[f'prob_{m}_{c}'] = probs[:, i]
        del model
        tf.keras.backend.clear_session()
    pred_df.to_csv(PRED_CSV, index=False)
    print("Prediksi per citra disimpan ke:", PRED_CSV)


pred_df = test_df[['image_id']].merge(pred_df, on='image_id', how='left', validate='one_to_one')
assert pred_df.filter(like='pred_').notna().all().all(), "Ada citra test tanpa prediksi."
y_true = pred_df['y_true'].values.astype(int)
N = len(pred_df)
correct = {m: (pred_df[f'pred_{m}'].values == y_true) for m in MODELS}

Prediksi per citra dimuat dari: /content/drive/MyDrive/Tesis_HAM10000/revisi_reviewer/prediksi_per_citra_test.csv


In [ ]:

# 3. TABEL IV DIHITUNG ULANG dari file prediksi yang sama

tabel4_rows = []
for m in MODELS:
    yp = pred_df[f'pred_{m}'].values
    probs = pred_df[[f'prob_{m}_{c}' for c in CLASS_NAMES]].values
    n_benar = int(correct[m].sum())
    lo, hi = proportion_confint(n_benar, N, alpha=ALPHA, method='wilson')
    yb = label_binarize(y_true, classes=np.arange(K))
    auc_macro = np.mean([roc_auc_score(yb[:, i], probs[:, i]) for i in range(K)])
    tabel4_rows.append({
        'Model': m, 'N_test': N, 'Jumlah_benar': n_benar,
        'Akurasi': n_benar / N, 'Akurasi_CI95_low': lo, 'Akurasi_CI95_high': hi,
        'Precision_macro': precision_score(y_true, yp, average='macro', zero_division=0),
        'Recall_macro': recall_score(y_true, yp, average='macro', zero_division=0),
        'F1_macro': f1_score(y_true, yp, average='macro', zero_division=0),
        'AUC_macro': auc_macro,
    })

    rp = f'{RESULTS_DIR}/{m}_result.json'
    if os.path.exists(rp):
        old_acc = json.load(open(rp))['accuracy']
        old_n = int(round(old_acc * N))
        status = "COCOK" if old_n == n_benar else "TIDAK COCOK -> perbarui Tabel IV"
        print(f"{m}: result.json lama = {old_n} benar, prediksi ulang = {n_benar} benar [{status}]")

tabel4 = pd.DataFrame(tabel4_rows)
tabel4.round(4).to_csv(f'{REVISI_DIR}/tabel4_dari_prediksi_per_citra.csv', index=False)
print("\nTABEL IV (versi terverifikasi):")
print(tabel4.round(4).to_string(index=False))

ResNet50: result.json lama = 1056 benar, prediksi ulang = 1056 benar [COCOK]
EfficientNetB0: result.json lama = 1058 benar, prediksi ulang = 1058 benar [COCOK]
VGG16: result.json lama = 1061 benar, prediksi ulang = 1061 benar [COCOK]

TABEL IV (versi terverifikasi):
         Model  N_test  Jumlah_benar  Akurasi  Akurasi_CI95_low  Akurasi_CI95_high  Precision_macro  Recall_macro  F1_macro  AUC_macro
      ResNet50    1502          1056   0.7031            0.6795             0.7256           0.5490        0.6409    0.5746     0.9353
EfficientNetB0    1502          1058   0.7044            0.6808             0.7269           0.5272        0.6219    0.5627     0.9233
         VGG16    1502          1061   0.7064            0.6829             0.7289           0.5717        0.6658    0.5858     0.9376


In [ ]:

# 4. MCNEMAR + VERIFIKASI KONTINGENSI  [R1, R2]

def kontingensi(a, b):
    ca, cb = correct[a], correct[b]
    n11 = int(np.sum(ca & cb));  n10 = int(np.sum(ca & ~cb))
    n01 = int(np.sum(~ca & cb)); n00 = int(np.sum(~ca & ~cb))

    assert n11 + n10 + n01 + n00 == N
    assert n11 + n10 == ca.sum() and n11 + n01 == cb.sum()
    assert n10 - n01 == int(ca.sum()) - int(cb.sum()), "Diskordan tidak konsisten dengan Tabel IV!"
    return n11, n10, n01, n00


pairs = [('ResNet50', 'EfficientNetB0'), ('ResNet50', 'VGG16'), ('EfficientNetB0', 'VGG16')]


NASKAH_N_BENAR = {'ResNet50': None, 'EfficientNetB0': None, 'VGG16': None}
NASKAH_DISKORDAN = {
    ('ResNet50', 'EfficientNetB0'): (133, 138),
    ('ResNet50', 'VGG16'):          (162, 164),
    ('EfficientNetB0', 'VGG16'):    (168, 171),
}

print("\n=== CEK SILANG NASKAH vs PREDIKSI PER CITRA ===")
for a, b in pairs:
    n11, n10, n01, n00 = kontingensi(a, b)
    nk10, nk01 = NASKAH_DISKORDAN.get((a, b), (None, None))
    print(f"\n{a} vs {b}")
    print(f"  Dihitung ulang : n10={n10}, n01={n01}, n10-n01={n10 - n01}, "
          f"selisih benar Tabel IV={int(correct[a].sum()) - int(correct[b].sum())}")
    if nk10 is not None:
        print(f"  Di naskah      : n10={nk10}, n01={nk01}, n10-n01={nk10 - nk01}"
              f"  -> {'sama' if (nk10, nk01) == (n10, n01) else 'BERBEDA, ganti dengan angka terverifikasi'}")
    if NASKAH_N_BENAR[a] is not None and NASKAH_N_BENAR[b] is not None and nk10 is not None:
        harap = NASKAH_N_BENAR[a] - NASKAH_N_BENAR[b]
        print(f"  Konsistensi internal naskah: n10-n01={nk10 - nk01} vs selisih Tabel IV={harap}"
              f" -> {'konsisten' if harap == nk10 - nk01 else 'TIDAK KONSISTEN (ini temuan reviewer)'}")


=== CEK SILANG NASKAH vs PREDIKSI PER CITRA ===

ResNet50 vs EfficientNetB0
  Dihitung ulang : n10=162, n01=164, n10-n01=-2, selisih benar Tabel IV=-2
  Di naskah      : n10=133, n01=138, n10-n01=-5  -> BERBEDA, ganti dengan angka terverifikasi

ResNet50 vs VGG16
  Dihitung ulang : n10=133, n01=138, n10-n01=-5, selisih benar Tabel IV=-5
  Di naskah      : n10=162, n01=164, n10-n01=-2  -> BERBEDA, ganti dengan angka terverifikasi

EfficientNetB0 vs VGG16
  Dihitung ulang : n10=168, n01=171, n10-n01=-3, selisih benar Tabel IV=-3
  Di naskah      : n10=168, n01=171, n10-n01=-3  -> sama


In [ ]:

# 5. CI 95% SELISIH AKURASI  [R3]

def newcombe_paired_ci(n11, n10, n01, n00, alpha=ALPHA):
    """Newcombe (1998) method 10: hybrid score CI untuk selisih proporsi berpasangan."""
    n = n11 + n10 + n01 + n00
    k1, k2 = n11 + n10, n11 + n01
    p1, p2 = k1 / n, k2 / n
    d = p1 - p2
    l1, u1 = proportion_confint(k1, n, alpha=alpha, method='wilson')
    l2, u2 = proportion_confint(k2, n, alpha=alpha, method='wilson')
    denom = np.sqrt(float(k1) * (n01 + n00) * k2 * (n10 + n00))
    phi = 0.0 if denom == 0 else (n11 * n00 - n10 * n01) / denom
    dl = np.sqrt(max(0.0, (p1 - l1) ** 2 - 2 * phi * (p1 - l1) * (u2 - p2) + (u2 - p2) ** 2))
    du = np.sqrt(max(0.0, (u1 - p1) ** 2 - 2 * phi * (u1 - p1) * (p2 - l2) + (p2 - l2) ** 2))
    return d, d - dl, d + du


def _lesion_sums(a, b):
    g = pd.DataFrame({'lesion': pred_df['lesion_id'],
                      'ca': correct[a].astype(int), 'cb': correct[b].astype(int)})
    g = g.groupby('lesion').agg(sa=('ca', 'sum'), sb=('cb', 'sum'), n=('ca', 'size'))
    return g['sa'].values, g['sb'].values, g['n'].values


def bootstrap_cluster_ci(a, b, alpha=ALPHA, B=5000):
    """Bootstrap dengan resampling LESI (bukan citra) -> menghormati korelasi antar citra satu lesi."""
    sa, sb, nn = _lesion_sums(a, b)
    idx = rng.integers(0, len(sa), size=(B, len(sa)))
    diffs = (sa[idx].sum(1) - sb[idx].sum(1)) / nn[idx].sum(1)
    return np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])


def permutation_cluster_test(a, b, n_perm=10000):
    """Uji permutasi sign-flip tingkat lesi (robustness untuk McNemar)."""
    sa, sb, _ = _lesion_sums(a, b)
    d = sa - sb
    obs = d.sum()
    signs = rng.choice([-1, 1], size=(n_perm, len(d)))
    perm = (signs * d).sum(1)
    return (np.sum(np.abs(perm) >= abs(obs)) + 1) / (n_perm + 1)

In [ ]:


MARGIN_UTAMA = 0.03
MARGIN_SENSITIVITAS = [0.02, 0.03, 0.05]


def tost_paired(n11, n10, n01, n00, margin, alpha=ALPHA):
    n = n11 + n10 + n01 + n00
    d = (n10 - n01) / n
    se = np.sqrt(max(n10 + n01 - (n10 - n01) ** 2 / n, 1e-12)) / n
    p_lower = 1 - norm.cdf((d + margin) / se)
    p_upper = norm.cdf((d - margin) / se)
    return max(p_lower, p_upper)


def ekuivalensi(a, b, margin, alpha=ALPHA):
    n11, n10, n01, n00 = kontingensi(a, b)
    p_tost = tost_paired(n11, n10, n01, n00, margin, alpha)
    _, lo90, hi90 = newcombe_paired_ci(n11, n10, n01, n00, alpha=2 * alpha)
    blo90, bhi90 = bootstrap_cluster_ci(a, b, alpha=2 * alpha)
    ekuiv = (p_tost < alpha) and (-margin < lo90 and hi90 < margin) and (-margin < blo90 and bhi90 < margin)
    return {'margin': margin, 'p_TOST': p_tost,
            'CI90_newcombe': (lo90, hi90), 'CI90_bootstrap_lesi': (blo90, bhi90),
            'ekuivalen': bool(ekuiv)}

In [ ]:

# 7. EKSEKUSI ANALISIS STATISTIK + KALIMAT INTERPRETASI
rows = []
for a, b in pairs:
    n11, n10, n01, n00 = kontingensi(a, b)
    table = [[n11, n10], [n01, n00]]
    p_exact = mcnemar(table, exact=True).pvalue
    chi = mcnemar(table, exact=False, correction=True)
    d, lo, hi = newcombe_paired_ci(n11, n10, n01, n00)
    blo, bhi = bootstrap_cluster_ci(a, b)
    p_perm = permutation_cluster_test(a, b)
    eq = ekuivalensi(a, b, MARGIN_UTAMA)
    rows.append({
        'Pasangan': f'{a} vs {b}', 'A': a, 'B': b,
        'n11_keduanya_benar': n11, 'n10_A_benar_B_salah': n10,
        'n01_A_salah_B_benar': n01, 'n00_keduanya_salah': n00,
        'Akurasi_A': (n11 + n10) / N, 'Akurasi_B': (n11 + n01) / N,
        'Selisih_akurasi_A_min_B': d,
        'CI95_newcombe_low': lo, 'CI95_newcombe_high': hi,
        'CI95_bootstrap_lesi_low': blo, 'CI95_bootstrap_lesi_high': bhi,
        'McNemar_chi2_cc': chi.statistic, 'p_McNemar_chi2_cc': chi.pvalue,
        'p_McNemar_exact': p_exact,
        'p_permutasi_lesi': p_perm,
        'Margin_ekuivalensi': MARGIN_UTAMA, 'p_TOST': eq['p_TOST'],
        'CI90_newcombe_low': eq['CI90_newcombe'][0], 'CI90_newcombe_high': eq['CI90_newcombe'][1],
        'Ekuivalen_dalam_margin': eq['ekuivalen'],
    })

stat_df = pd.DataFrame(rows)
stat_df['p_McNemar_exact_Holm'] = multipletests(stat_df['p_McNemar_exact'], method='holm')[1]
stat_df.to_csv(f'{REVISI_DIR}/mcnemar_ci_tost_terverifikasi.csv', index=False)
print("\n=== HASIL STATISTIK TERVERIFIKASI ===")
print(stat_df.round(4).T.to_string())

# Analisis sensitivitas margin
sens = []
for a, b in pairs:
    for mg in MARGIN_SENSITIVITAS:
        e = ekuivalensi(a, b, mg)
        sens.append({'Pasangan': f'{a} vs {b}', 'Margin': mg, 'p_TOST': e['p_TOST'],
                     'CI90_low': e['CI90_newcombe'][0], 'CI90_high': e['CI90_newcombe'][1],
                     'Ekuivalen': e['ekuivalen']})
sens_df = pd.DataFrame(sens)
sens_df.to_csv(f'{REVISI_DIR}/sensitivitas_margin_ekuivalensi.csv', index=False)
print("\n=== SENSITIVITAS MARGIN EKUIVALENSI ===")
print(sens_df.round(4).to_string(index=False))


def kalimat_hasil(r):
    pp = lambda x: f"{x * 100:.2f}"
    s = (f"{r['A']} vs {r['B']}: accuracy difference = {pp(r['Selisih_akurasi_A_min_B'])} percentage points "
         f"(95% CI {pp(r['CI95_newcombe_low'])} to {pp(r['CI95_newcombe_high'])}; "
         f"lesion-level bootstrap 95% CI {pp(r['CI95_bootstrap_lesi_low'])} to {pp(r['CI95_bootstrap_lesi_high'])}). ")
    if r['p_McNemar_exact_Holm'] >= ALPHA:
        s += (f"There was no statistically significant difference in error proportions "
              f"(exact McNemar p = {r['p_McNemar_exact']:.3f}, Holm-adjusted p = {r['p_McNemar_exact_Holm']:.3f}); "
              f"the available evidence is insufficient to reject the null hypothesis of equal error proportions. ")
    else:
        s += (f"The difference was statistically significant "
              f"(exact McNemar p = {r['p_McNemar_exact']:.3f}, Holm-adjusted p = {r['p_McNemar_exact_Holm']:.3f}). ")
    if r['Ekuivalen_dalam_margin']:
        s += (f"Using a pre-specified equivalence margin of ±{pp(r['Margin_ekuivalensi'])} percentage points, "
              f"the two models were statistically equivalent (TOST p = {r['p_TOST']:.3f}).")
    else:
        s += (f"Equivalence within ±{pp(r['Margin_ekuivalensi'])} percentage points could not be established "
              f"(TOST p = {r['p_TOST']:.3f}); therefore, this result should not be interpreted as the models being equivalent.")
    return s


with open(f'{REVISI_DIR}/kalimat_hasil_untuk_naskah.txt', 'w') as f:
    for _, r in stat_df.iterrows():
        k = kalimat_hasil(r)
        print("\n" + k)
        f.write(k + "\n\n")


=== HASIL STATISTIK TERVERIFIKASI ===
                                                   0                  1                        2
Pasangan                  ResNet50 vs EfficientNetB0  ResNet50 vs VGG16  EfficientNetB0 vs VGG16
A                                           ResNet50           ResNet50           EfficientNetB0
B                                     EfficientNetB0              VGG16                    VGG16
n11_keduanya_benar                               894                923                      890
n10_A_benar_B_salah                              162                133                      168
n01_A_salah_B_benar                              164                138                      171
n00_keduanya_salah                               282                308                      273
Akurasi_A                                     0.7031             0.7031                   0.7044
Akurasi_B                                     0.7044             0.7064                 

In [ ]:

# 8. BUKTI KOMPARATIF LEAKAGE: lesion-level vs image-level split  [R6]

img_train, img_temp = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
img_val, img_test = train_test_split(img_temp, test_size=0.50, stratify=img_temp['label'], random_state=SEED)
img_train, img_val, img_test = [d.reset_index(drop=True) for d in (img_train, img_val, img_test)]


def statistik_kebocoran(tr, te, nama):
    shared = set(tr['lesion_id']) & set(te['lesion_id'])
    bocor = te['lesion_id'].isin(shared)
    return {'Skema': nama, 'Lesi_test_juga_di_train': len(shared),
            'Citra_test_bocor': int(bocor.sum()), 'Persen_citra_test_bocor': 100 * bocor.mean()}


leak_overlap = pd.DataFrame([
    statistik_kebocoran(train_df, test_df, 'Lesion-level (penelitian ini)'),
    statistik_kebocoran(img_train, img_test, 'Image-level (pembanding)'),
])
print("\n=== OVERLAP LESION_ID TRAIN-TEST ===")
print(leak_overlap.round(2).to_string(index=False))
leak_overlap.to_csv(f'{REVISI_DIR}/overlap_lesion_train_test.csv', index=False)


=== OVERLAP LESION_ID TRAIN-TEST ===
                        Skema  Lesi_test_juga_di_train  Citra_test_bocor  Persen_citra_test_bocor
Lesion-level (penelitian ini)                        0                 0                     0.00
     Image-level (pembanding)                      512               543                    36.13


In [ ]:

import imagehash
from PIL import Image

HASH_CSV = f'{REVISI_DIR}/phash_semua_citra.csv'
if os.path.exists(HASH_CSV):
    hash_df = pd.read_csv(HASH_CSV, dtype={'phash': str})
else:
    print("Menghitung perceptual hash (beberapa menit)...")
    hash_df = pd.DataFrame({'image_id': df['image_id'],
                            'phash': [str(imagehash.phash(Image.open(p).convert('RGB'))) for p in df['path']]})
    hash_df.to_csv(HASH_CSV, index=False)
hmap = dict(zip(hash_df['image_id'], hash_df['phash']))

HAMMING_THRESHOLD = 6   # semakin kecil semakin ketat (0 = identik)


def near_duplicates(tr, te, thr=HAMMING_THRESHOLD, chunk=100):
    h_tr = np.array([int(hmap[i], 16) for i in tr['image_id']], dtype=np.uint64)
    h_te = np.array([int(hmap[i], 16) for i in te['image_id']], dtype=np.uint64)
    tr_les = tr['lesion_id'].values
    out = []
    for s in range(0, len(h_te), chunk):
        x = np.bitwise_xor(h_te[s:s + chunk, None], h_tr[None, :])
        dist = np.unpackbits(x.view(np.uint8), axis=-1).reshape(x.shape[0], x.shape[1], 64).sum(-1)
        j = dist.argmin(axis=1)
        for k in range(len(j)):
            out.append((dist[k, j[k]], tr_les[j[k]]))
    dmin = np.array([o[0] for o in out])
    same_les = np.array([o[1] for o in out]) == te['lesion_id'].values
    near = dmin <= thr
    return {'Citra_test_near_dup': int(near.sum()),
            'Near_dup_lesi_sama': int((near & same_les).sum()),
            'Near_dup_lesi_beda': int((near & ~same_les).sum())}


dup_rows = []
for nama, tr, te in [('Lesion-level (penelitian ini)', train_df, test_df),
                     ('Image-level (pembanding)', img_train, img_test)]:
    r = near_duplicates(tr, te); r['Skema'] = nama; dup_rows.append(r)
dup_df = pd.DataFrame(dup_rows)[['Skema', 'Citra_test_near_dup', 'Near_dup_lesi_sama', 'Near_dup_lesi_beda']]
print(f"\n=== NEAR-DUPLICATE TRAIN-TEST (Hamming pHash <= {HAMMING_THRESHOLD}) ===")
print(dup_df.to_string(index=False))
dup_df.to_csv(f'{REVISI_DIR}/near_duplicate_train_test.csv', index=False)



=== NEAR-DUPLICATE TRAIN-TEST (Hamming pHash <= 6) ===
                        Skema  Citra_test_near_dup  Near_dup_lesi_sama  Near_dup_lesi_beda
Lesion-level (penelitian ini)                  107                   0                 107
     Image-level (pembanding)                  142                  24                 118


In [ ]:

RUN_LEAKAGE_RETRAIN = True
LEAKAGE_MODELS = ['EfficientNetB0']

EPOCHS_STAGE1, EPOCHS_STAGE2 = 15, 20
LR_STAGE1, LR_STAGE2 = 1e-3, 1e-5

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.15), layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1), layers.RandomContrast(0.15),
], name="augmentation")


def make_dataset(dataframe, preprocess_fn, training=False):
    labels = tf.keras.utils.to_categorical(dataframe['label'].values, num_classes=K)
    ds = tf.data.Dataset.from_tensor_slices((dataframe['path'].values, labels))
    ds = ds.map(load_and_decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


def build_model(base_fn):
    base = base_fn(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = False
    inp = layers.Input(shape=(224, 224, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(K, activation='softmax')(x)
    return models.Model(inp, out), base


def cbs(ckpt, backup):
    return [callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
            callbacks.ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, verbose=0),
            callbacks.BackupAndRestore(backup_dir=backup)]


def train_two_stage(model_name, tr, va, tag):
    fpath = f'{LEAK_DIR}/final_{model_name}_{tag}.keras'
    if os.path.exists(fpath):
        print(f"{model_name} ({tag}) sudah ada, dimuat.")
        return tf.keras.models.load_model(fpath)
    cfg = MODEL_CONFIGS[model_name]
    cw = compute_class_weight('balanced', classes=np.arange(K), y=tr['label'].values)
    cw = {i: w for i, w in enumerate(cw)}
    tr_ds = make_dataset(tr, cfg['preprocess'], training=True)
    va_ds = make_dataset(va, cfg['preprocess'])
    metrics = ['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=True)]


    s1_done = f'{LEAK_DIR}/stage1_selesai_{model_name}_{tag}.keras'
    if os.path.exists(s1_done):
        print(f"Tahap 1 {model_name} ({tag}) sudah selesai, dimuat -> langsung ke Tahap 2.")
        model = tf.keras.models.load_model(s1_done)
        base = next(l for l in model.layers if isinstance(l, tf.keras.Model))
    else:
        model, base = build_model(cfg['fn'])
        model.compile(optimizer=optimizers.Adam(learning_rate=LR_STAGE1),
                      loss='categorical_crossentropy', metrics=metrics)
        model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS_STAGE1, class_weight=cw,
                  callbacks=cbs(f'{LEAK_DIR}/best_{model_name}_{tag}_s1.keras',
                                f'{LEAK_DIR}/backup_{model_name}_{tag}_s1'), verbose=1)
        model.save(s1_done)
        print("Tahap 1 tersimpan:", s1_done)

    ft_at = int(len(base.layers) * 0.7)
    base.trainable = True
    for l in base.layers[:ft_at]:
        l.trainable = False
    for l in base.layers[ft_at:]:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    model.compile(optimizer=optimizers.Adam(learning_rate=LR_STAGE2),
                  loss='categorical_crossentropy', metrics=metrics)

    if hasattr(model, '_initial_epoch'):
        model._initial_epoch = None
    model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS_STAGE2, class_weight=cw,
              callbacks=cbs(f'{LEAK_DIR}/best_{model_name}_{tag}_s2.keras',
                            f'{LEAK_DIR}/backup_{model_name}_{tag}_s2'), verbose=1)
    model.save(fpath)
    return model


def ringkas(nama, m, yt, yp, mask=None):
    if mask is not None:
        yt, yp = yt[mask], yp[mask]
    nb = int((yt == yp).sum()); n = len(yt)
    lo, hi = proportion_confint(nb, n, alpha=ALPHA, method='wilson') if n else (np.nan, np.nan)
    return {'Skema/subset': nama, 'Model': m, 'N': n, 'Akurasi': nb / n if n else np.nan,
            'CI95_low': lo, 'CI95_high': hi,
            'F1_macro': f1_score(yt, yp, average='macro', zero_division=0) if n else np.nan}


if RUN_LEAKAGE_RETRAIN:
    leak_rows = []
    for m in LEAKAGE_MODELS:

        leak_rows.append(ringkas('Lesion-level: seluruh test', m, y_true, pred_df[f'pred_{m}'].values))

        model = train_two_stage(m, img_train, img_val, 'imagelevel')
        probs = model.predict(make_dataset(img_test, MODEL_CONFIGS[m]['preprocess']), verbose=0)
        yp_img = probs.argmax(1); yt_img = img_test['label'].values
        bocor = img_test['lesion_id'].isin(set(img_train['lesion_id'])).values
        leak_rows.append(ringkas('Image-level: seluruh test', m, yt_img, yp_img))
        leak_rows.append(ringkas('Image-level: subset BOCOR (lesi ada di train)', m, yt_img, yp_img, bocor))
        leak_rows.append(ringkas('Image-level: subset BERSIH (lesi tidak di train)', m, yt_img, yp_img, ~bocor))
        del model; tf.keras.backend.clear_session()

    leak_df = pd.DataFrame(leak_rows)
    leak_df.to_csv(f'{REVISI_DIR}/eksperimen_kebocoran_komparatif.csv', index=False)
    print("\n=== EKSPERIMEN KOMPARATIF KEBOCORAN ===")
    print(leak_df.round(4).to_string(index=False))


print("\nSemua luaran revisi tersimpan di:", REVISI_DIR)

Tahap 1 EfficientNetB0 (imagelevel) sudah selesai, dimuat -> langsung ke Tahap 2.
Epoch 16/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 253s 707ms/step - accuracy: 0.6506 - auc: 0.9406 - loss: 0.6748 - val_accuracy: 0.6551 - val_auc: 0.9326 - val_loss: 0.8084 - learning_rate: 5.0000e-06
Epoch 17/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 157s 541ms/step - accuracy: 0.6662 - auc: 0.9426 - loss: 0.6460 - val_accuracy: 0.6704 - val_auc: 0.9338 - val_loss: 0.7782 - learning_rate: 5.0000e-06
Epoch 18/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 154s 534ms/step - accuracy: 0.6656 - auc: 0.9464 - loss: 0.6069 - val_accuracy: 0.6704 - val_auc: 0.9341 - val_loss: 0.7843 - learning_rate: 5.0000e-06
Epoch 19/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 150s 528ms/step - accuracy: 0.6769 - auc: 0.9494 - loss: 0.5908 - val_accuracy: 0.6711 - val_auc: 0.9348 - val_loss: 0.7690 - learning_rate: 5.0000e-06
Epoch 20/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 153s 531ms/step - accuracy: 0.6598 - auc: 0.9342 - loss: 0.7621 - val_accuracy: 0.6631 - val_auc: 0.9337 

In [ ]:

# 9. ANALISIS KEBOCORAN DENGAN KONTROL KOMPOSISI KELAS

from sklearn.metrics import balanced_accuracy_score

m = 'EfficientNetB0'
IMG_PRED_CSV = f'{REVISI_DIR}/prediksi_imagelevel_{m}.csv'
if os.path.exists(IMG_PRED_CSV):
    ip = pd.read_csv(IMG_PRED_CSV)
else:
    model = tf.keras.models.load_model(f'{LEAK_DIR}/final_{m}_imagelevel.keras')
    probs = model.predict(make_dataset(img_test, MODEL_CONFIGS[m]['preprocess']), verbose=0)
    ip = img_test[['image_id', 'lesion_id', 'dx', 'label']].copy()
    ip['pred'] = probs.argmax(1)
    ip['bocor'] = ip['lesion_id'].isin(set(img_train['lesion_id']))
    ip.to_csv(IMG_PRED_CSV, index=False)
    del model; tf.keras.backend.clear_session()
ip['benar'] = ip['pred'] == ip['label']
ip['Subset'] = np.where(ip['bocor'], 'BOCOR', 'BERSIH')

komp = pd.crosstab(ip['dx'], ip['Subset'], normalize='columns').mul(100).round(1)
print("=== (a) KOMPOSISI KELAS (%) ===\n", komp)


rows = []
for c in CLASS_NAMES:
    r = {'Kelas': c}
    for s in ['BOCOR', 'BERSIH']:
        sub = ip[(ip['Subset'] == s) & (ip['dx'] == c)]
        r[f'n_{s}'] = len(sub)
        r[f'Recall_{s}'] = sub['benar'].mean() if len(sub) else np.nan
    r['Selisih_BOCOR_min_BERSIH'] = r['Recall_BOCOR'] - r['Recall_BERSIH']
    rows.append(r)
per_kelas = pd.DataFrame(rows)
print("\n=== (b) RECALL PER KELAS ===\n", per_kelas.round(3).to_string(index=False))


w = ip['dx'].value_counts(normalize=True)
valid = per_kelas.dropna(subset=['Recall_BOCOR', 'Recall_BERSIH'])
ww = w[valid['Kelas']].values / w[valid['Kelas']].sum()
ringkasan = pd.DataFrame([
    {'Skema/subset': 'Lesion-level (penelitian ini)',
     'Balanced_accuracy': balanced_accuracy_score(y_true, pred_df[f'pred_{m}'])},
    {'Skema/subset': 'Image-level: seluruh test',
     'Balanced_accuracy': balanced_accuracy_score(ip['label'], ip['pred'])},
    {'Skema/subset': 'Image-level: BOCOR',
     'Balanced_accuracy': balanced_accuracy_score(ip.loc[ip.bocor, 'label'], ip.loc[ip.bocor, 'pred']),
     'Akurasi_terstandardisasi_kelas': float((valid['Recall_BOCOR'].values * ww).sum())},
    {'Skema/subset': 'Image-level: BERSIH',
     'Balanced_accuracy': balanced_accuracy_score(ip.loc[~ip.bocor, 'label'], ip.loc[~ip.bocor, 'pred']),
     'Akurasi_terstandardisasi_kelas': float((valid['Recall_BERSIH'].values * ww).sum())},
])
print("\n=== (c) METRIK TAHAN KOMPOSISI KELAS ===\n", ringkasan.round(4).to_string(index=False))

komp.to_csv(f'{REVISI_DIR}/kebocoran_komposisi_kelas.csv')
per_kelas.to_csv(f'{REVISI_DIR}/kebocoran_recall_per_kelas.csv', index=False)
ringkasan.to_csv(f'{REVISI_DIR}/kebocoran_metrik_tahan_kelas.csv', index=False)

=== (a) KOMPOSISI KELAS (%) ===
 Subset  BERSIH  BOCOR
dx                   
akiec      3.2    3.3
bcc        2.4    9.9
bkl        8.5   15.3
df         1.0    1.3
mel        6.4   19.5
nv        76.9   49.4
vasc       1.6    1.3

=== (b) RECALL PER KELAS ===
 Kelas  n_BOCOR  Recall_BOCOR  n_BERSIH  Recall_BERSIH  Selisih_BOCOR_min_BERSIH
akiec       18         0.444        31          0.452                    -0.007
  bcc       54         0.648        23          0.391                     0.257
  bkl       83         0.663        82          0.634                     0.029
   df        7         0.714        10          0.500                     0.214
  mel      106         0.613        61          0.574                     0.039
   nv      268         0.437       738          0.805                    -0.368
 vasc        7         0.857        15          0.867                    -0.010

=== (c) METRIK TAHAN KOMPOSISI KELAS ===
                  Skema/subset  Balanced_accuracy  Akura